# Sesión 05 - Lab 2: Combinación de DataFrames con tablas de referencia

Este laboratorio combina `dbassociate.silver.clientes_crm` (Lab 1) con dos tablas de referencia del mismo CRM: `direcciones_crm` (qué direcciones tiene cada cliente) y `direcciones_referencia` (los datos de cada dirección). A diferencia de la tabla transaccional de clientes, estas dos ya llegan curadas del CRM, así que se cargan directo a Silver, sin pasar por una limpieza de Bronze. Dos clientes (`4007` y `4015`) tienen más de una dirección registrada (`Principal` y `Envio`): es el caso de fan-out de este laboratorio. Un cliente (`4013`) no tiene ninguna dirección registrada, para practicar el `LEFT JOIN`.

## Verificación del entorno

In [0]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_05")

print("Filas en clientes_crm (Silver):")
spark.sql("SELECT COUNT(*) AS filas FROM dbassociate.silver.clientes_crm").show()

## Cargar las tablas de referencia del CRM

`direcciones_crm` y `direcciones_referencia` ya llegan curadas del sistema origen (sin nulos, sin duplicados): se cargan directo a Silver, sin el paso de limpieza que sí hizo falta para los datos transaccionales de clientes.

In [0]:
df_direcciones_crm_raw = spark.read.csv(
    "/Volumes/dbassociate/default/vol_landing/sesion_05/direcciones_crm.csv",
    header=True,
    inferSchema=True,
)
df_direcciones_crm_raw.write.mode("overwrite").saveAsTable("dbassociate.silver.direcciones_crm")

df_direcciones_referencia_raw = spark.read.csv(
    "/Volumes/dbassociate/default/vol_landing/sesion_05/direcciones_referencia.csv",
    header=True,
    inferSchema=True,
)
df_direcciones_referencia_raw.write.mode("overwrite").saveAsTable("dbassociate.silver.direcciones_referencia")

# guardamos las tablas en silver

print("Direcciones por cliente:", df_direcciones_crm_raw.count())
print("Direcciones de referencia:", df_direcciones_referencia_raw.count())

## Lab 2A: Inner join para enriquecer con la dirección del cliente

`direcciones_crm` es una tabla puente: por cada cliente puede haber una fila (si tiene una sola dirección) o más de una (si tiene varias). En este dataset, los clientes `4007` y `4015` tienen dos direcciones cada uno.

Cuando hacés `INNER JOIN` por `cliente_id`, Spark busca, para cada cliente, todas las filas de `direcciones_crm` que tengan ese mismo `cliente_id`, y arma una fila de resultado por cada coincidencia que encuentra. Si un cliente tiene una sola dirección, sale una fila. Si tiene dos, salen dos filas, aunque el cliente sea uno solo. A esa multiplicación de filas se le llama fan-out, y es justo lo que vas a ver abajo: el conteo total de filas después del join queda por encima del número de clientes.

In [0]:
from pyspark.sql.functions import col

df_clientes = spark.table("dbassociate.silver.clientes_crm")
df_direcciones_cliente = spark.table("dbassociate.silver.direcciones_crm")

# join por ID
df_inner = df_clientes.join(df_direcciones_cliente, on="cliente_id", how="inner")

print("Clientes en Silver:", df_clientes.count())
print("Filas tras el INNER JOIN:", df_inner.count())

# tenemos clientes con duplicados
df_inner.groupBy("cliente_id").count().filter(col("count") > 1).show(truncate=False)

## Lab 2B: Left join para conservar clientes sin dirección registrada

Un `INNER JOIN` descarta a cualquier cliente sin una fila coincidente en `direcciones_crm` (`4013`, en este dataset). Un `LEFT JOIN` le dice a Spark "quiero conservar todos los clientes pase lo que pase, y si alguno no tiene dirección, dejá esas columnas vacías (`null`) en vez de sacarlo del resultado". Si el objetivo es un catálogo completo de clientes, es la opción correcta.

In [0]:
# clientes sin direcciones
df_left = df_clientes.join(df_direcciones_cliente, on="cliente_id", how="left")

clientes_sin_direccion = df_left.filter(col("direccion_id").isNull())


print("Clientes en Silver:", df_clientes.count())
print("Filas tras el LEFT JOIN:", df_left.count())
clientes_sin_direccion.select("cliente_id", "nombre", "apellido").show(truncate=False)

## Lab 2C: Evitar el fan-out desde el join, agregando una segunda condición

A diferencia del Lab 2A, acá el resultado **no** va a tener clientes duplicados: la idea de esta celda es justamente evitar que el fan-out llegue a pasar, no mostrarlo.

En el Lab 2A el join decía, en criollo, "juntá cliente con dirección, donde coincida el `cliente_id`". Como un cliente puede tener más de una dirección, esa instrucción no alcanza para saber cuál dirección traer, y Spark trae todas las que coincidan (por eso se duplica). Acá le agregamos una segunda condición: "juntá cliente con dirección, donde coincida el `cliente_id` **y además** la dirección sea la `Principal`". Con esa aclaración de más, cada cliente tiene como mucho una dirección que cumple las dos condiciones a la vez, así que el resultado nunca se duplica.

In [0]:
df_solo_principal = df_clientes.join(
    df_direcciones_cliente,
    on=(df_clientes["cliente_id"] == df_direcciones_cliente["cliente_id"]) ## se colocar un alias
    & (df_direcciones_cliente["tipo_direccion"] == "Principal"), # solo nos quedamos con las principales
    how="left",
).drop(df_direcciones_cliente["cliente_id"])

print("Filas tras el join con multiples condiciones:", df_solo_principal.count())
df_solo_principal.select("cliente_id", "nombre", "apellido", "tipo_direccion").show(10, truncate=False)

## Lab 2D: Broadcast join con la tabla de referencia direcciones_referencia

`direcciones_referencia` es una tabla de referencia chica frente a `clientes_crm`.

En criollo: cuando Spark hace un join, por default reparte pedacitos de las dos tablas entre las máquinas del cluster (eso es el shuffle) para que cada una pueda emparejar lo que le tocó. Si una de las dos tablas es chica, ese reparto es un desperdicio: sale más rápido darle una copia completa de la tabla chica a cada máquina de una sola vez (eso es el broadcast), así ninguna tiene que esperar a que le lleguen pedazos de la tabla grande. `broadcast(direcciones_referencia)` le pide a Spark exactamente eso, y `.explain()` muestra en el plan de ejecución que efectivamente usó esa estrategia (`BroadcastHashJoin`).

In [0]:
from pyspark.sql.functions import broadcast

df_direcciones_ref = spark.table("dbassociate.silver.direcciones_referencia")

# se puede utilizar un configuración global a nivel cluster o de celda para que el broadcast se aplique a df de maximos 10MB u otro peso que consideremos.
df_clientes_enriquecido = df_solo_principal.join(broadcast(df_direcciones_ref), on="direccion_id", how="left")

df_clientes_enriquecido.select(
    "cliente_id", "nombre", "apellido", "ciudad", "region"
).show(10, truncate=False)

df_clientes_enriquecido.explain()

## Lab 2E: Cross join, demostración controlada

En criollo: mientras que un `INNER`/`LEFT JOIN` empareja filas que coinciden en algo (una llave), `crossJoin()` no le pide que coincida nada, junta cada fila de una tabla con cada fila de la otra, todas contra todas. Sirve para generar combinaciones exhaustivas (por ejemplo, todas las combinaciones posibles entre un grupo chico de clientes y un grupo chico de asesores comerciales), pero crece de forma cuadrática: cruzar dos tablas de miles de filas cada una genera millones de filas de salida por accidente. Por eso Spark no permite un `join()` sin condición a menos que sea explícitamente un `crossJoin()`.

In [0]:
df_top_clientes = df_clientes.select("cliente_id", "nombre", "apellido").limit(3)
df_asesores = df_clientes.select("asesor_comercial").distinct().limit(3)

# cross join, genera todas las combinaciones posibles entre las tablas
df_cross = df_top_clientes.crossJoin(df_asesores)

print("Clientes:", df_top_clientes.count(), "| Asesores:", df_asesores.count())
print("Filas del cross join:", df_cross.count())
df_cross.show(truncate=False)

## Lab 2F: Union y union all

En criollo: pensalo como una pila de hojas. Un join agrega columnas nuevas al costado de cada fila (la hoja se hace más ancha). Un union agrega filas nuevas abajo de las que ya había (la pila se hace más alta), pero las columnas tienen que ser las mismas de los dos lados.

`union`/`unionByName` combina DataFrames **verticalmente** (apila filas, mismas columnas), a diferencia de un join, que combina **horizontalmente** (agrega columnas de otra tabla por una llave). `unionByName` alinea por nombre de columna en vez de por posición, más seguro si el orden de columnas difiere entre ambos DataFrames. Ni `union` ni `unionByName` deduplican: si las dos fuentes se solapan, hay que deduplicar después.

In [0]:
# unionAll no permite duplicados, union si permite.

df_direcciones_principal = df_direcciones_cliente.filter(col("tipo_direccion") == "Principal")
df_direcciones_envio = df_direcciones_cliente.filter(col("tipo_direccion") == "Envio")

df_todas_direcciones = df_direcciones_principal.unionByName(df_direcciones_envio)

print("Principal:", df_direcciones_principal.count())
print("Envio:", df_direcciones_envio.count())
print("Union sin deduplicar:", df_todas_direcciones.count())

df_direcciones_unicas = df_todas_direcciones.dropDuplicates(["cliente_id", "direccion_id"])
print("Despues de dropDuplicates:", df_direcciones_unicas.count())

## Lab 2G: Resolver el fan-out con una función de ventana

Lab 2C evitó el fan-out filtrando dentro del join. En criollo, acá hacemos lo contrario: dejamos que el join produzca todas las filas (el `df_inner` del Lab 2A, con clientes duplicados) y arreglamos el problema después. `row_number()` sobre una ventana numera las filas de cada cliente según el orden que le indiquemos, y nos quedamos solo con la número 1 (mismo mecanismo del Lab 1D, aplicado ahora para resolver un fan-out en vez de una reingesta duplicada). Es la alternativa a preferir cuando la regla de prioridad es más compleja que un único valor fijo de `tipo_direccion`.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

ventana_direccion = Window.partitionBy("cliente_id").orderBy(col("tipo_direccion"))

df_una_direccion_por_cliente = (
    df_inner
    .withColumn("prioridad", row_number().over(ventana_direccion))
    .filter(col("prioridad") == 1) # tome el primero por cliente segun el orden de la dirección, este tipo dirección es texto y tiene un orden alfabetico
    .drop("prioridad")
)

print("Filas tras el INNER JOIN (con fan-out):", df_inner.count())
print("Filas tras resolver el fan-out con row_number:", df_una_direccion_por_cliente.count())

## Escribir el resultado enriquecido en Silver

In [0]:
df_clientes_enriquecido.write.mode("overwrite").saveAsTable("dbassociate.silver.clientes_enriquecido")

print("Filas escritas en clientes_enriquecido:", df_clientes_enriquecido.count())

## Consulta de validación

In [0]:
spark.sql("""
    SELECT region, ciudad, COUNT(*) AS clientes
    FROM dbassociate.silver.clientes_enriquecido
    GROUP BY region, ciudad
    ORDER BY clientes DESC
""").show(truncate=False)

## Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.direcciones_crm")
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.direcciones_referencia")
spark.sql("DROP TABLE IF EXISTS dbassociate.silver.clientes_enriquecido")

print("Tablas temporales de este laboratorio eliminadas.")